<a href="https://colab.research.google.com/github/Minhaj401/nlp/blob/main/sentiment_analysis_simplernn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from keras.datasets import imdb
from keras.models import Sequential
from keras.layers import Embedding, SimpleRNN, Dense
from keras.preprocessing.sequence import pad_sequences

In [3]:
(X_train,y_train),(X_test,y_test) = imdb.load_data()

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [4]:
X_train = pad_sequences(X_train,padding='post',maxlen=500)
X_test = pad_sequences(X_test,padding='post',maxlen=500)

In [5]:
y_test

array([0, 1, 1, ..., 0, 0, 0])

In [8]:
model = Sequential()
model.add(Embedding(100000, 2, input_length=500))
model.add(SimpleRNN(32,return_sequences=False))
model.add(Dense(1, activation='sigmoid'))

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['acc'])
history = model.fit(X_train, y_train,epochs=5,validation_data=(X_test,y_test))

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 101s 125ms/step - acc: 0.4992 - loss: 0.6944 - val_acc: 0.4991 - val_loss: 0.6944
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 100s 128ms/step - acc: 0.4971 - loss: 0.6952 - val_acc: 0.5034 - val_loss: 0.6932
Epoch 3/5


### Test the model with your own input

Now you can enter a sentence or phrase, and the model will predict its sentiment.

In [ ]:
# Get user input
user_input_text = input("Enter a sentence for sentiment prediction: ")
print(f"You entered: {user_input_text}")

Enter a sentence for sentiment prediction: not good not bad not goodbad
You entered: not good not bad not goodbad


In [ ]:
# Preprocess the user input
# The IMDB dataset does not use a Tokenizer object like the one from keras.preprocessing.text.
# Instead, it provides a word_index mapping.
word_index = imdb.get_word_index()

# Prepare a mapping for unknown words and special tokens
# The indices in the IMDB dataset are typically offset by 3:
# 0: padding, 1: start of sequence, 2: unknown, 3: unused
indexed_review = []
for word in user_input_text.lower().split():
    # Get the index for the word, default to 0 if not found (which becomes 3 after offset)
    # Add 3 to the word index to match the format used by imdb.load_data()
    idx = word_index.get(word, 0) + 3
    # Ensure index is within the Embedding layer's vocabulary size (10000)
    if idx >= 10000:
        indexed_review.append(2) # Map to unknown (index 2) if outside vocabulary size
    else:
        indexed_review.append(idx)

# Wrap in a list for batch processing
user_sequence_list = [indexed_review]

# Pad the sequence to match the input_length of the model (maxlen=50)
# Make sure to use the same padding style ('post')
user_padded_sequence = pad_sequences(user_sequence_list, padding='post', maxlen=50)

print("Preprocessed sequence:")
print(user_padded_sequence)

Preprocessed sequence:
[[24 52 24 78 24  3  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0]]


In [ ]:
# Make a prediction
prediction = model.predict(user_padded_sequence)

# The output is a probability. For binary classification, > 0.5 can be considered positive.
if prediction[0][0] > 0.5:
    sentiment = 'Positive'
else:
    sentiment = 'Negative'

print(f"\nPrediction probability: {prediction[0][0]:.4f}")
print(f"Predicted sentiment: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step

Prediction probability: 0.1076
Predicted sentiment: Negative
